In [1]:
import os
import pandas as pd
import numpy as np

# LTAU 하이퍼파라미터(ltau_bins, ltau_k) 12개 조합 간 fold별 성능 차이가
# 통계적으로 유의미한지 ANOVA로 검정한다.
# 참고: run_cue_ltau_param_1.sh / run_cue_ltau_param_2.sh
cue_base = '/data/rlawlsgurjh/work/gitlab/CUE/cue/results'

datasets = ['DIS', 'INH']
targets = ['PI', 'Ssel', 'WS2', 'RS2']
metrics_of_interest = ['rmse', 'r2', 'pcc']

# run_cue_ltau_param_1.sh / _2.sh의 LTAU_BINS_LIST / LTAU_K_LIST와 동일하게 맞춘다.
LTAU_BINS_LIST = [1250, 2500, 5000, 10000]
LTAU_K_LIST = [3, 5, 7]

In [2]:
def load_fold_metrics(dataset, target, bins, k, task_type='regression'):
    """predictions_all.csv(5-fold 예측 전체)를 읽어 fold별 rmse/r2/pcc를 계산한다.

    주의: train.py::aggregate_metrics()의 fold 루프(정렬된 Fold 순서, GroundTruth/예측값에
    대한 동일한 isfinite 마스킹, 동일한 get_metrics 호출)를 한 글자도 다르지 않게 재현해야
    fold별 값의 평균이 기존 metrics.csv의 rmse/r2/pcc와 소수점까지 정확히 일치한다.
    """
    model_dir = os.path.join(cue_base, dataset, target, 'v1', f'LTAU_BIN{bins}_K{k}')
    path = os.path.join(model_dir, 'predictions_all.csv')
    if not os.path.exists(path):
        return None

    df_ys = pd.read_csv(path)
    folds = sorted(df_ys['Fold'].unique())
    model_name = 'mlp'

    rows = []
    for fold in folds:
        fd = df_ys[df_ys['Fold'] == fold]

        y_true = fd['GroundTruth'].values.astype(float)
        y_pred = fd[model_name].values.astype(float)

        mask = np.isfinite(y_true) & np.isfinite(y_pred)
        y_true_f = y_true[mask]
        y_pred_f = y_pred[mask]

        if y_true_f.size == 0:
            continue

        met = get_metrics(y_true_f, y_pred_f, task_type=task_type)
        rows.append({'Fold': fold, 'rmse': met['rmse'], 'r2': met['r2'], 'pcc': met['pcc']})

    return pd.DataFrame(rows) if rows else None

import sys
sys.path.append('/data/rlawlsgurjh/work/gitlab/CUE/cmfe')
from cue.utils.metric import get_metrics

raw_rows = []
for d_name in datasets:
    for target in targets:
        for bins in LTAU_BINS_LIST:
            for k in LTAU_K_LIST:
                df_fold = load_fold_metrics(d_name, target, bins, k)
                if df_fold is None:
                    continue
                for _, r in df_fold.iterrows():
                    raw_rows.append({
                        'Dataset': d_name, 'Target': target, 'BIN': bins, 'K': k, 'Fold': r['Fold'],
                        'rmse': r['rmse'], 'r2': r['r2'], 'pcc': r['pcc'],
                    })

df_folds = pd.DataFrame(raw_rows)
print(f"✅ Fold별 데이터 로드 완료: {len(df_folds)}개의 행이 수집되었습니다 (기대값: 2 x 4 x 12 x 5 = 480).")

✅ Fold별 데이터 로드 완료: 480개의 행이 수집되었습니다 (기대값: 2 x 4 x 12 x 5 = 480).


In [3]:
from scipy import stats

def run_anova(df_folds, dataset, target, metric):
    """지정한 dataset/target/metric에 대해 12개의 (BIN, K) 조합별 fold 값들(5개씩)을
    독립 표본으로 삼아 일원분산분석(one-way ANOVA)을 수행한다.

    Returns:
        Optional[Dict]: {'F_stat': ..., 'p_value': ...} 혹은 그룹이 부족하면 None.
    """
    groups = []
    for bins in LTAU_BINS_LIST:
        for k in LTAU_K_LIST:
            vals = df_folds[
                (df_folds['Dataset'] == dataset) & (df_folds['Target'] == target) &
                (df_folds['BIN'] == bins) & (df_folds['K'] == k)
            ][metric].dropna().values
            if len(vals) > 0:
                groups.append(vals)

    if len(groups) < 2:
        return None

    f_stat, p_value = stats.f_oneway(*groups)
    return {'F_stat': float(f_stat), 'p_value': float(p_value)}


table_save_dir = os.path.join(cue_base, 'table', 'ltau_param_anova')
os.makedirs(table_save_dir, exist_ok=True)

# DIS -> dis(dissociation), INH -> inh(inhibition)
dataset_filenames = {'DIS': 'dis.csv', 'INH': 'inh.csv'}

anova_results = {}
for d_name in datasets:
    dataset_rows = []
    for target in targets:
        for metric in metrics_of_interest:
            result = run_anova(df_folds, d_name, target, metric)
            if result is None:
                continue
            anova_results[(d_name, target, metric)] = result

            row = {
                'Target': target,
                'Metric': metric,
                'F_stat': round(result['F_stat'], 3),
                'p_value': round(result['p_value'], 4),
                'significant (p<0.05)': result['p_value'] < 0.05,
            }
            dataset_rows.append(row)

            print(f"[{d_name} / {target} / {metric}] F={row['F_stat']}, p={row['p_value']}, significant={row['significant (p<0.05)']}")

    if not dataset_rows:
        continue

    dataset_table = pd.DataFrame(dataset_rows)
    save_path = os.path.join(table_save_dir, dataset_filenames[d_name])
    dataset_table.to_csv(save_path, index=False, encoding='utf-8-sig', sep='\t')
    print(f"\n✅ 저장 완료: {save_path}")
    print(dataset_table.to_string(index=False))
    print()

[Kd / PI / rmse] F=0.001, p=1.0, significant=False
[Kd / PI / r2] F=0.002, p=1.0, significant=False
[Kd / PI / pcc] F=0.0, p=1.0, significant=False
[Kd / Ssel / rmse] F=0.0, p=1.0, significant=False
[Kd / Ssel / r2] F=0.0, p=1.0, significant=False
[Kd / Ssel / pcc] F=0.0, p=1.0, significant=False
[Kd / WS(alpha=2) / rmse] F=0.0, p=1.0, significant=False
[Kd / WS(alpha=2) / r2] F=0.0, p=1.0, significant=False
[Kd / WS(alpha=2) / pcc] F=0.0, p=1.0, significant=False
[Kd / RS(k=2) / rmse] F=0.0, p=1.0, significant=False
[Kd / RS(k=2) / r2] F=0.0, p=1.0, significant=False
[Kd / RS(k=2) / pcc] F=0.0, p=1.0, significant=False

✅ 저장 완료: /data/rlawlsgurjh/work/gitlab/CUE/cmfe/results/table/ltau_param_anova/dis.csv
     Target Metric  F_stat  p_value  significant (p<0.05)
         PI   rmse   0.001      1.0                 False
         PI     r2   0.002      1.0                 False
         PI    pcc   0.000      1.0                 False
       Ssel   rmse   0.000      1.0                 